# Role C: BUSI box experiment and analysis

Run P1_00 first to download datasets and weights. Uses the committed BUSI split. MedSAM has seen BUSI during training; report it separately from unseen datasets. No output scores are prefilled. Run U-Net separately if its baseline is required.

In [ ]:
# Parameters (edit before Run all if needed)
CONFIG = "configs/c_busi_box.yaml"
MODELS = ["sam_vit_b", "sam_vit_h", "medsam"]

In [ ]:
# SETUP: identical in every notebook (generated by scripts/build_notebooks.py).
import base64
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/geekRoy0126/promptshift.git"
REF = "main"  # branch, tag or commit to run
# Storage root on Google Drive (data, weights, results, checkpoints). Change it here if needed.
DRIVE_ROOT = "/content/drive/MyDrive/cs5330/cs5330/promptshift"

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["PROMPTSHIFT_ROOT"] = DRIVE_ROOT  # read by promptshift.paths in every script
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
    REPO = Path("/content/promptshift")
    try:  # optional: the repo is public; GH_TOKEN is only needed if it is made private again
        _token = userdata.get("GH_TOKEN")
    except Exception:  # secret missing or notebook access not granted
        _token = ""
    _auth = base64.b64encode(f"x-access-token:{_token}".encode()).decode() if _token else ""

    def _git(*args, auth=False):
        pre = ["-c", f"http.extraHeader=Authorization: Basic {_auth}"] if auth and _auth else []
        r = subprocess.run(["git", *pre, "-C", str(REPO), *args], capture_output=True, text=True)
        if r.returncode:
            msg = r.stderr or r.stdout
            for s in (_token, _auth):
                msg = msg.replace(s, "***") if s else msg
            raise RuntimeError(f"git {args[0]} failed: {msg}")

    REPO.mkdir(parents=True, exist_ok=True)
    if not (REPO / ".git").is_dir():
        _git("init", "-q")
    _git("fetch", "-q", "--depth", "1", REPO_URL, REF, auth=True)
    _git("reset", "-q", "--hard", "FETCH_HEAD")  # throwaway clone; all outputs live on Drive
    del _token, _auth
    _pip = [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)]
    r = subprocess.run(_pip, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("pip install -e failed (log above)")
else:  # local CPU run: use the checkout this notebook lives in
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONUNBUFFERED"] = "1"
# scripts must import this checkout's code, whatever else is installed
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), os.environ.get("PYTHONPATH", "")])


def run(*args, check=True):
    """Run a repo script with live output; stop the notebook if it fails. Returns the output."""
    cmd = [sys.executable, *map(str, args)]
    print("$ python", " ".join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = []
    while chunk := p.stdout.read1(4096):
        text = chunk.decode(errors="replace")
        out.append(text)
        sys.stdout.write(text)
        sys.stdout.flush()
    if p.wait() and check:
        raise RuntimeError(f"exit code {p.returncode}: python {' '.join(cmd[1:])}")
    return "".join(out)


def check_gpu():
    import torch

    if IN_COLAB and not torch.cuda.is_available():
        raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.")
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")


def git_out(*args):
    r = subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True)
    return r.stdout.strip()


SHA = git_out("rev-parse", "--short", "HEAD")
print(f"repo {REPO} @ {SHA} (Colab: {IN_COLAB})")

In [ ]:
check_gpu()
run("scripts/run_experiment.py", "--config", CONFIG, "--models", *MODELS)

## Statistics and figures
Missing baselines are explicitly recorded in analysis_status.json.

In [ ]:
run("scripts/analyze_results.py", "--config", CONFIG)

## Failure cases
Requires real weights. Review causes manually in review.csv.

In [ ]:
run("scripts/make_failure_cases.py", "--config", CONFIG, "--top", "6")

## Export
Download this archive before ending the session.

In [ ]:
import shutil

from promptshift.paths import get_paths

paths = get_paths()
archive = shutil.make_archive(
    str(paths.root / "Role_C_BUSI"), "zip",
    root_dir=paths.summary_results / "c_busi_box")
if IN_COLAB:
    from google.colab import files
    files.download(archive)
else:
    print(archive)